In [0]:
%run "../00-Common/01. Env Config"

In [0]:
#Defing target table for gold layer storage
target_table=f"{catalog_name}.{gold_schema}.dim_results"

In [0]:
# reading relevant tables form silver tables for join

results_df=(spark.table(f"{catalog_name}.{silver_schema}.results")
                    .withColumn("session_type", F.lit("Race"))
                    .drop("race_name", "race_date","ingestion_timestamp","source_file")
)


sprints_df=(spark.table(f"{catalog_name}.{silver_schema}.sprints")
                    .withColumn("session_type", F.lit("Sprint"))
                    .drop("race_name", "race_date","ingestion_timestamp","source_file")
)

In [0]:
#unioinizing the tables

results_sprints_unionized=(
                            results_df.unionByName(sprints_df)
)


### Adding Derived columns

1 is_win-> Will indicate if driver won the race

2 is_podium-> Will indicate if the driver came either 1st, 2nd, or 3rd

3 has_points->> will simply flag drivers who scored more points

In [0]:
results_sprints_final= (results_sprints_unionized
                                            .withColumn("is_win",F.col("position")==1)
                                            .withColumn("is_podium",F.col("position").between(1,3))
                                            .withColumn("has_points",F.col("points")>0)

)

In [0]:
#Writting final table to Gold layer

(
    results_sprints_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)


In [0]:
display(results_sprints_final.filter(F.col("season")==2025))